# 第12回：改善実験を小さく回す

**今日の問い：改善した理由を後から説明できる実験とは何か。**

上から順に実行してください。`TRY`は全員、`CHANGE`は値を1つ変える練習、
`CHALLENGE`は余裕がある人向けです。`DEEP DIVE`は経験者や自習向けの発展です。
分からないコードは、セル全体ではなく気になる数行をM365 Copilotへ貼って相談します。


In [ ]:
from pathlib import Path

def find_repo_root(start=Path.cwd()):
    for candidate in [start, *start.parents]:
        if (candidate / "pyproject.toml").exists():
            return candidate
    raise FileNotFoundError("pyproject.tomlがある勉強会フォルダ内で実行してください")

ROOT = find_repo_root()
DATA = ROOT / "data"
print("教材フォルダ:", ROOT)


## この回でできるようになること

- 変更を1要素に限定した比較を設計し、実験ログを関数で残す
- RandomizedSearchCVで探索し、ネストCVで楽観の少ない推定を得る
- 並べ替え重要度の信頼区間とエラー分析から次の仮説を選ぶ

### 進み方

`CORE`は同期90分で扱う本線、`DEEP DIVE`は時間があれば扱う深掘り、
`SELF-STUDY`は任意自習です。すべて終わらなくても次回へ進めます。
経験者は`CORE`を早めに終え、`DEEP DIVE`を5人で分担して読むと深まります。

### 先に押さえる言葉

- 実験ログ：変更・条件・結果・解釈を残す記録
- ランダム探索：候補を無作為に試すハイパーパラメータ探索
- ネストCV：探索と評価を分けて過大評価を防ぐ交差検証
- 信頼区間：推定値の不確かさを表す幅
- 再現性：同じ手順で同じ結果を得られる性質

> **実行前の30秒予想**：今日の問いに、今の言葉で仮の答えを書いてから始めます。


In [ ]:
import pandas as pd

df = pd.read_csv(DATA / "compound_experiments.csv")
print(f"{len(df)}行 × {len(df.columns)}列")
df.head()


In [ ]:
from sklearn.model_selection import cross_validate, StratifiedKFold
from sklearn.impute import SimpleImputer
from sklearn.pipeline import make_pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.inspection import permutation_importance

features = ["temperature_c", "reaction_time_h", "concentration_m", "molecular_weight", "logp", "tpsa"]
X, y = df[features], df["active"]
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)


## TRY：1要素だけ変えて記録する


In [ ]:
rows = []
for depth in [3, 6, None]:
    model = make_pipeline(SimpleImputer(strategy="median"), RandomForestClassifier(n_estimators=150, max_depth=depth, random_state=42))
    scores = cross_validate(model, X, y, cv=cv, scoring="f1", return_train_score=True)
    rows.append({"実験名": f"depth={depth}", "変更点": "max_depthのみ",
                 "学習F1": scores["train_score"].mean(), "検証F1平均": scores["test_score"].mean(),
                 "検証F1標準偏差": scores["test_score"].std()})
experiment_log = pd.DataFrame(rows)
experiment_log.round(3)


## 実験ログの最小項目

- 実験名 / 変えたもの（1つ） / 固定した比較条件 / 結果の平均とばらつき / 気づき / 次の仮説

Copilotには案を出してもらい、優先順位と予測時点の妥当性は人が判断します。


## DEEP DIVE：ランダム探索・ネストCV・重要度の区間

手作業の総当たりではなく、探索と評価を分けて楽観の少ない推定を得ます。


In [ ]:
from sklearn.model_selection import RandomizedSearchCV

pipe = make_pipeline(SimpleImputer(strategy="median"), RandomForestClassifier(random_state=42))
param_dist = {
    "randomforestclassifier__n_estimators": [100, 200, 300],
    "randomforestclassifier__max_depth": [3, 4, 6, None],
    "randomforestclassifier__min_samples_leaf": [1, 2, 4],
    "randomforestclassifier__max_features": ["sqrt", "log2", None],
}
search = RandomizedSearchCV(pipe, param_dist, n_iter=10, cv=cv, scoring="f1", random_state=42)
search.fit(X, y)
print("最良設定:", search.best_params_)
print("探索内での最良CV F1:", round(search.best_score_, 3))


In [ ]:
from sklearn.model_selection import cross_val_score

outer = StratifiedKFold(5, shuffle=True, random_state=7)
nested = cross_val_score(search, X, y, cv=outer, scoring="f1")
print("ネストCV外側F1:", nested.round(3))
print("楽観の少ない推定:", round(nested.mean(), 3), "±", round(nested.std(), 3), " ← 探索内スコアより低いのが普通")


### 並べ替え重要度は区間で読む

平均だけでなくばらつきを見て、0を跨ぐ列は寄与があるとは言い切れません（評価はholdoutで行います）。


In [ ]:
from sklearn.model_selection import train_test_split

X_fit, X_holdout, y_fit, y_holdout = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)
best = make_pipeline(SimpleImputer(strategy="median"), RandomForestClassifier(n_estimators=200, max_depth=6, random_state=42)).fit(X_fit, y_fit)
perm = permutation_importance(best, X_holdout, y_holdout, scoring="f1", n_repeats=30, random_state=42)
importance = pd.DataFrame({
    "特徴量": features,
    "重要度平均": perm.importances_mean,
    "下限(平均-2SD)": perm.importances_mean - 2 * perm.importances_std,
}).sort_values("重要度平均", ascending=False)
importance["0を跨ぐ"] = importance["下限(平均-2SD)"] <= 0
importance.round(4)


## よくある誤り

- 同時に複数要素を変える
- 探索に使った分割で最終性能も報告する
- 悪化した実験を記録から消す

## SELF-STUDY（任意・30〜60分）

- RandomizedSearchCVの最良設定を、ネストCVの外側スコアで確かめる
- 並べ替え重要度を20反復で計算し、区間が0を跨ぐ列を挙げる

成果は完成したコードでなくても、予想・変更点・出力・解釈を4行で残せば十分です。

## 振り返りチェック

1. 1要素だけ変える理由は何か
2. ネストCVは何を防ぐか
3. 重要度の区間が0を跨ぐとどう解釈するか

答えに詰まった項目が、次に見返す場所です。暗記ではなくNotebookの該当セルを指せればOKです。
